In [ ]:
# # 读取fa文件，统计不同序列出现的次数以及对应的overall_confidence的均值，最小值，最大值，保存到csv文件中
# import pandas as pd
# import os
# from collections import defaultdict
# from Bio.PDB import PDBParser
# from Bio.SeqUtils import seq1

# with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed/PDB.list', 'r') as f:
#     pdbs = [line.strip() for line in f.readlines()]

# for pdb in pdbs:
#     method_path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/LigandMPNN-Output-pred_dimer/AF3-2000-0.1T/'

#     pdb_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/PepSet_AF3_pass"
#     pdb_list = os.listdir(pdb_path)
#     # print(pdb_list)
#     if f'{pdb}.pdb' in pdb_list:
#         #读取pdb，将A链的单字母序列保存到变量sequence中，注意pdb只有ATOM或HETATM行
#         parser = PDBParser(QUIET=True)
#         structure = parser.get_structure(pdb, f'{pdb_path}/{pdb}.pdb')
#         for model in structure:
#             for chain in model:
#                 if chain.id == 'A':
#                     pro_sequence = ''
#                     for residue in chain:
#                         if residue.id[0] == ' ':
#                             one_letter_resname = seq1(residue.get_resname())
#                             pro_sequence += one_letter_resname

#     fa_file = f'{method_path}/{pdb}/seqs/{pdb}.fa'
#     sequences = []
#     confidences = defaultdict(list)
#     with open(fa_file, 'r') as f:
#         for i, line in enumerate(f):
#             if i >= 2:
#                 if line.startswith('>'):
#                     parts = line.strip().split(',')
#                     overall_confidence = float(parts[4].split('=')[1])
#                     ligand_confidence = float(parts[5].split('=')[1])
#                     seq_rec = float(parts[6].split('=')[1])
#                 else:
#                     seq = line.strip()
#                     sequences.append(seq)
#                     confidences[seq].append([overall_confidence, ligand_confidence, seq_rec])

#     sequence_stats = []
#     for seq, conf_list in confidences.items():
#         count = len(conf_list)
#         avg_overall_confidence = round(sum([conf[0] for conf in conf_list]) / count, 3)
#         min_overall_confidence = min([conf[0] for conf in conf_list])
#         max_overall_confidence = max([conf[0] for conf in conf_list])
#         avg_ligand_confidence = round(sum([conf[1] for conf in conf_list]) / count, 3)
#         min_ligand_confidence = min([conf[1] for conf in conf_list])
#         max_ligand_confidence = max([conf[1] for conf in conf_list])
#         avg_seq_rec = round(sum([conf[2] for conf in conf_list]) / count, 3)
#         sequence_stats.append((pro_sequence, seq, count, avg_overall_confidence, min_overall_confidence, max_overall_confidence, avg_ligand_confidence, min_ligand_confidence, max_ligand_confidence, avg_seq_rec))
#     df = pd.DataFrame(
#         sequence_stats,
#         columns=['Pro_Sequence', 'Pep_Sequence', 'Count', 'Average_Overall_Confidence', 'Min_Overall_Confidence', 'Max_Overall_Confidence', 'Average_Ligand_Confidence', 'Min_Ligand_Confidence', 'Max_Ligand_Confidence', 'Average_Seq_Rec']
#     )

#     df = df.sort_values(by='Max_Overall_Confidence', ascending=False)
#     df.index = range(1, len(df) + 1)
#     # os.makedirs(f'{method_path}/ranked_by_Max_Overall_Confidence', exist_ok=True)
    
#     df.to_csv(f'{method_path}/{pdb}/ranked_by_Max_Overall_Confidence.csv', index=True)


In [4]:
import json
import os
import pandas as pd


af3_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict"
mpnn_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/data/AF3-2000-0.1T"

pdbs = [pdb for pdb in sorted(os.listdir(mpnn_path)) if os.path.isdir(os.path.join(mpnn_path, pdb))]

os.makedirs(af3_path, exist_ok=True)
json_temple_path = "template.json"

for pdb in pdbs:
    for seed in ['seed42']:
        df = pd.read_csv(f'{mpnn_path}/{pdb}/{seed}/ranked_by_Max_Overall_Confidence.csv')
        pep_sequences = df["Pep_Sequence"].tolist()
        pro_sequence = df["Pro_Sequence"].tolist()[0]
        msa_pro_path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/MSA_pro_all_PepSet_dimer/{pdb}'

        for i, pep_sequence in enumerate(pep_sequences[:100]):
            os.makedirs(f"{af3_path}/inputs", exist_ok=True)
            os.makedirs(f"{af3_path}/outputs", exist_ok=True)

            #根据template.json的内容以及fasta文件，编写json文件，将fasta中的pro序列替换template.json中的pro_sequence，将fasta中的pep序列替换template.json中的pep_sequence，保存为新的json文件
            with open(json_temple_path, 'r') as file:
                job = json.load(file)

            seq_pep = pep_sequence
            seq_pro = pro_sequence
            
            # 如果msa存在，则替换msa路径和pairing_db，如果不存在，则不添加msa路径和pairing_db
            job['sequences'][0]['protein']['pairedMsaPath'] = msa_pro_path + "/pairing.a3m"
            job['sequences'][0]['protein']['unpairedMsaPath'] = msa_pro_path + "/non_pairing.a3m"

            job['name'] = pdb + "_" + str(i+1)
            job['sequences'][0]['protein']['sequence'] = seq_pro
            job['sequences'][1]['protein']['sequence'] = seq_pep
            job['modelSeeds'] = [42,43,44,45,46]

            with open(f'{af3_path}/inputs/{pdb}_{i+1}.json', 'w') as f:
                f.write(json.dumps(job, indent=4))
